# Jet Tagging Solution

This notebook contains the project solution for binary classification, multi-class classification, and momentum decorrelation on the ATLAS-like jet-tagging dataset. The original assignment text is preserved separately in `instructions.ipynb`; reusable loading, feature, sampling, model, and metric code lives under `src/physics_applications_of_ai/`.

## Approach

The solution uses gradient-boosted decision trees on engineered jet features. Tasks 1 and 2 use global substructure variables plus relative constituent features where useful. Task 3 removes direct four-momentum inputs and balances the samples in shared `(pt, mass)` bins, then audits residual dependence on the momentum variables.

In [ ]:
import pandas
from matplotlib import pyplot
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import train_test_split

from physics_applications_of_ai.data import DISPLAY_LABELS
from physics_applications_of_ai.datasets import (
    make_binary_dataset,
    make_decorrelated_dataset,
    make_multiclass_dataset,
)
from physics_applications_of_ai.evaluation import (
    binary_metrics,
    multiclass_metrics,
    prediction_momentum_eta_squared,
    probability_momentum_correlations,
)
from physics_applications_of_ai.models import make_hist_gradient_boosting_classifier

RANDOM_STATE = 7


def train_classifier(X, y, *, max_iter: int, l2_regularization: float):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
    )
    model = make_hist_gradient_boosting_classifier(
        max_iter=max_iter,
        l2_regularization=l2_regularization,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)
    predictions = probabilities.argmax(axis=1)
    return model, X_test, y_test, predictions, probabilities

## Task 1: Binary Classification

Two balanced binary classifiers are trained: quark/gluon vs W/Z and quark/gluon vs top. The table reports accuracy, F1, ROC-AUC, and average precision on held-out test splits.

In [ ]:
task1_evaluations = []

for positive_label, display_label in [("wz", "W/Z"), ("top", "top")]:
    X, y = make_binary_dataset(positive_label, n_per_class=100_000, random_state=RANDOM_STATE)
    model, X_test, y_test, _, probabilities = train_classifier(
        X, y, max_iter=180, l2_regularization=0.02
    )
    positive_probabilities = probabilities[:, 1]
    predictions = (positive_probabilities >= 0.5).astype(int)

    print(f"\nquark/gluon vs {display_label}")
    print(classification_report(y_test, predictions, target_names=["quark/gluon", display_label]))

    task1_evaluations.append({
        "task": f"quark/gluon vs {display_label}",
        "model": model,
        "X_test": X_test,
        "y_test": y_test,
        "predictions": predictions,
        "probabilities": positive_probabilities,
        **binary_metrics(y_test, predictions, positive_probabilities),
    })

task1_results = pandas.DataFrame([
    {metric: evaluation[metric] for metric in ["task", "accuracy", "f1", "roc_auc", "average_precision"]}
    for evaluation in task1_evaluations
]).set_index("task")
task1_results

In [ ]:
fig, axes = pyplot.subplots(2, 2, figsize=(12, 9))
for row_index, evaluation in enumerate(task1_evaluations):
    positive_label = evaluation["task"].split(" vs ")[1]
    ConfusionMatrixDisplay.from_predictions(
        evaluation["y_test"],
        evaluation["predictions"],
        display_labels=["quark/gluon", positive_label],
        normalize="true",
        ax=axes[row_index, 0],
        colorbar=False,
    )
    axes[row_index, 0].set_title(f"{evaluation['task']} confusion matrix")

    RocCurveDisplay.from_predictions(
        evaluation["y_test"], evaluation["probabilities"], ax=axes[row_index, 1]
    )
    axes[row_index, 1].set_title(f"{evaluation['task']} ROC curve")

pyplot.tight_layout()

## Task 2: Multi-Class Classification

The multi-class model uses a balanced sample from all three classes and combines global substructure features with relative constituent features.

In [ ]:
X_task2, y_task2 = make_multiclass_dataset(
    n_per_class=75_000, random_state=RANDOM_STATE
)
task2_model, X_task2_test, y_task2_test, task2_predictions, task2_probabilities = train_classifier(
    X_task2, y_task2, max_iter=220, l2_regularization=0.02
)

print(classification_report(y_task2_test, task2_predictions, target_names=DISPLAY_LABELS))

task2_results = pandas.DataFrame([
    multiclass_metrics(y_task2_test, task2_predictions, task2_probabilities)
])
task2_results

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_task2_test,
    task2_predictions,
    display_labels=DISPLAY_LABELS,
    normalize="true",
    cmap="Blues",
)
pyplot.title("Task 2 normalized confusion matrix")
pyplot.tight_layout()

## Task 3: Momentum Decorrelation

The decorrelated model omits direct `pt`, `eta`, `phi`, and `mass` inputs. It uses unitless substructure ratios, relative constituent geometry, and equalized sampling in shared `(pt, mass)` bins. The metric tables show both classification performance and residual momentum dependence.

In [ ]:
X_task3, y_task3, task3_momenta = make_decorrelated_dataset(
    n_bins=5,
    max_per_bin_per_class=4_000,
    min_per_bin_per_class=25,
    random_state=RANDOM_STATE,
)
(
    X_task3_train,
    X_task3_test,
    y_task3_train,
    y_task3_test,
    task3_momenta_train,
    task3_momenta_test,
) = train_test_split(
    X_task3,
    y_task3,
    task3_momenta,
    test_size=0.25,
    stratify=y_task3,
    random_state=RANDOM_STATE,
)

task3_model = make_hist_gradient_boosting_classifier(
    max_iter=220, l2_regularization=0.08, random_state=RANDOM_STATE
)
task3_model.fit(X_task3_train, y_task3_train)
task3_probabilities = task3_model.predict_proba(X_task3_test)
task3_predictions = task3_probabilities.argmax(axis=1)

print(classification_report(y_task3_test, task3_predictions, target_names=DISPLAY_LABELS))

task3_results = pandas.DataFrame([{
    **multiclass_metrics(y_task3_test, task3_predictions, task3_probabilities),
    "n_train": len(y_task3_train),
    "n_test": len(y_task3_test),
    "n_features": X_task3.shape[1],
}])
task3_probability_momentum_correlations = probability_momentum_correlations(
    task3_probabilities, task3_momenta_test
)
task3_prediction_momentum_eta_squared = prediction_momentum_eta_squared(
    task3_predictions, task3_momenta_test
)

display(task3_results)
display(task3_probability_momentum_correlations)
display(task3_prediction_momentum_eta_squared)

In [ ]:
fig, axes = pyplot.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_task3_test,
    task3_predictions,
    display_labels=DISPLAY_LABELS,
    normalize="true",
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Task 3 normalized confusion matrix")

task3_momenta_test.assign(predicted_class=task3_predictions).boxplot(
    column="mass", by="predicted_class", ax=axes[1]
)
axes[1].set_title("Jet mass by predicted class after decorrelation")
axes[1].set_xlabel("predicted class index")
axes[1].set_ylabel("mass [GeV]")
fig.suptitle("")
pyplot.tight_layout()

## Summary

The binary classifiers perform strongly, the three-class classifier remains useful but harder, and the decorrelated model shows the expected tradeoff: lower classification performance but much weaker dependence on global momentum variables.